# 第 05 课 · 圆柱几何、浸入边界近似与体积惩罚

> **学习模式：**独立编写 JAX CFD 求解器 | VS Code / Jupyter | 从公式到可运行代码

**本课目标**：从笛卡尔网格中创建圆柱掩膜，添加固体区域的速度约束，并理解其局限。

**前置知识**：第 04 课二维周期 Navier–Stokes。

**操作约定**：先逐单元执行，再尝试“实验与练习”；若改变了函数定义或网格设置，请重启内核并执行全部单元格。JAX 第一次调用包含编译时间。

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax import lax
jax.config.update("jax_enable_x64", True)
print("JAX:", jax.__version__, "devices:", jax.devices())

## 1. 在笛卡尔网格表示圆柱

圆柱横截面满足：

$$\chi(x,y)=\begin{cases}1,&(x-x_c)^2+(y-y_c)^2\le R^2,\\0,&\text{otherwise}.\end{cases}$$

直接使用不连续掩膜可能造成网格敏感性，教学求解器使用平滑掩膜：

$$\chi_\epsilon(x,y)=\frac12\left[1-\tanh\left(\frac{r-R}{\epsilon}\right)\right],\qquad r=\sqrt{(x-x_c)^2+(y-y_c)^2}. $$

注意本课程先使用**周期域**，不是均匀入口 / 压力出口的孤立圆柱算例。后续对结果的解释必须保留这一限制。

In [ ]:
def grid(nx=64,ny=32,Lx=8.,Ly=4.):
 dx,dy=Lx/nx,Ly/ny
 x=jnp.arange(nx)*dx; y=jnp.arange(ny)*dy
 X,Y=jnp.meshgrid(x,y,indexing='ij')
 return dx,dy,X,Y

In [ ]:
def geometry(nx,ny,dx,dy,cx=2,cy=2,r=.5,eps=None):
 eps=0.8*min(dx,dy) if eps is None else eps
 xu=(jnp.arange(nx)[:,None]+.5)*dx; yu=jnp.arange(ny)[None,:]*dy
 xv=jnp.arange(nx)[:,None]*dx; yv=(jnp.arange(ny)[None,:]+.5)*dy
 # simple nonperiodic distance OK when body away from edges
 ru=jnp.sqrt((xu-cx)**2+(yu-cy)**2)
 rv=jnp.sqrt((xv-cx)**2+(yv-cy)**2)
 masku=.5*(1-jnp.tanh((ru-r)/eps))
 maskv=.5*(1-jnp.tanh((rv-r)/eps))
 return masku,maskv

In [ ]:
nx,ny,Lx,Ly=96,48,8.0,4.0
dx,dy,X,Y=grid(nx,ny,Lx,Ly)
cx,cy,R=2.0,2.0,0.5
mask_u,mask_v=geometry(nx,ny,dx,dy,cx,cy,R)
fig,ax=plt.subplots(figsize=(9,3.4))
im=ax.imshow(np.asarray(mask_u).T,extent=(0,Lx,0,Ly),origin='lower',vmin=0,vmax=1,cmap='viridis',aspect='equal')
ax.set_title('Smoothed cylinder mask at u faces'); ax.set_xlabel('x');ax.set_ylabel('y'); fig.colorbar(im,ax=ax);plt.show()
print('diameter in grid spacings:',2*R/dx)
assert mask_u.shape==(nx,ny)

## 2. 体积惩罚法：用源项模拟固体速度条件

对于静止固体，增加惩罚力：

$$\frac{\partial\mathbf u}{\partial t}+\cdots
=-\nabla p+\nu\nabla^2\mathbf u
-\frac{\chi_\epsilon}{\eta}(\mathbf u-\mathbf u_s),\qquad \mathbf u_s=\mathbf 0.$$

$\eta$ 越小，速度越强烈地被拉向固体目标速度，但直接显式积分会变得刚性。我们对这个局部阻尼项作**半隐式更新**：

$$u_{\text{penal}}=\frac{u^*+(\Delta t/\eta)\chi u_s}{1+(\Delta t/\eta)\chi}.$$

这只是一个教学用的平滑体积惩罚近似：它能制造障碍物和尾流，但未严格保证圆柱表面无滑移，也不能直接得到标准高精度阻力系数。

In [ ]:
dt,eta=0.015,0.025
u_star=jnp.ones((nx,ny))
u_penal=u_star/(1+dt/eta*mask_u)
print('before mean in solid core:',float(jnp.mean(u_star[jnp.asarray(mask_u>0.95)])))
print('after mean in solid core:',float(jnp.mean(u_penal[jnp.asarray(mask_u>0.95)])))
fig,ax=plt.subplots(figsize=(9,3.4))
im=ax.imshow(np.asarray(u_penal).T,origin='lower',extent=(0,Lx,0,Ly),cmap='coolwarm',vmin=0,vmax=1,aspect='equal')
ax.set_title('One penalization substep (before pressure projection)');fig.colorbar(im,ax=ax);plt.show()

## 3. 结合压力投影，理解约束之间的竞争

惩罚步骤减小固体内速度，但一般会引入散度；压力投影将散度降至数值精度附近，却可能再次改变固体附近速度。因此**一次惩罚 + 一次投影不能严格同时满足实体壁面无滑移和处处不可压缩**。现代浸入边界、约束投影和体贴网格方法会更细致地处理这一问题。

In [ ]:
def div(u,v,dx,dy):
 return (u-jnp.roll(u,1,0))/dx+(v-jnp.roll(v,1,1))/dy

In [ ]:
def grad(p,dx,dy):
 return (jnp.roll(p,-1,0)-p)/dx,(jnp.roll(p,-1,1)-p)/dy

In [ ]:
def lap(q,dx,dy):
 return (jnp.roll(q,-1,0)+jnp.roll(q,1,0)-2*q)/(dx*dx)+(jnp.roll(q,-1,1)+jnp.roll(q,1,1)-2*q)/(dy*dy)

In [ ]:
def symbol(nx,ny,dx,dy):
 kx=2*jnp.pi*jnp.fft.fftfreq(nx,d=dx)[:,None]
 ky=2*jnp.pi*jnp.fft.fftfreq(ny,d=dy)[None,:]
 s=-4*jnp.sin(kx*dx/2)**2/dx**2-4*jnp.sin(ky*dy/2)**2/dy**2
 return jnp.where(s==0,1.,s)

In [ ]:
def solve_poisson(rhs,s):
 rhs_hat=jnp.fft.fftn(rhs)
 p=jnp.fft.ifftn(rhs_hat/s).real
 return p-p.mean()

In [ ]:
def project(u,v,dx,dy,s):
 p=solve_poisson(div(u,v,dx,dy),s)
 px,py=grad(p,dx,dy)
 return u-px,v-py,p

In [ ]:
S=symbol(nx,ny,dx,dy)
u_proj,v_proj,p=project(u_penal,jnp.zeros_like(u_penal),dx,dy,S)
print('max div after projection:',float(jnp.max(jnp.abs(div(u_proj,v_proj,dx,dy)))))
print('u mean inside body after projection:',float(jnp.sum(u_proj*mask_u)/jnp.sum(mask_u)))
assert float(jnp.max(jnp.abs(div(u_proj,v_proj,dx,dy))))<1e-10

## 4. 实验与练习

1. 改变 $R=0.25,0.5,0.75$，查看掩膜所占面积和圆柱分辨率。
2. 比较 $\epsilon=0.4\Delta x$ 与 $1.5\Delta x$ 时的边界平滑程度。
3. 比较 $\eta=0.2,0.025,0.005$ 的阻尼强度，解释“更小”为什么不一定等于“更准确”。

**本课通过条件**：能画出圆柱掩膜、实现惩罚步骤，并解释投影/无滑移的局限。